# NOVA 00 — Setup Check
Verifies GPU, HF token (secret `HF_TOKEN`), and repo access in ~1 minute.
Run this first, once.

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Verify HF token works and repos can be created under unixio
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
me = api.whoami()
print('Logged in as:', me['name'])
assert me['name'] == 'unixio', f"Expected unixio, got {me['name']}"


In [ ]:
# Create the four model repos (idempotent)
from huggingface_hub import create_repo
for repo in ['nova-obstacle-detection', 'nova-currency-detection',
             'nova-face-detection', 'nova-face-embedding']:
    create_repo(f'unixio/{repo}', repo_type='model', exist_ok=True,
                token=os.environ['HF_TOKEN'])
    print(f'ready: https://huggingface.co/unixio/{repo}')

In [ ]:
# Smoke-test the obstacle pipeline end-to-end on COCO128 (~3 min on T4).
# Proves: ultralytics install, GPU training, TFLite INT8 export.
!pip install -q ultralytics onnx2tf onnx
!python scripts/train_obstacle.py --data coco128.yaml --epochs 1 --imgsz 320 --batch 16